# 04 · Annual cycles and seasonal stripes

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/04-seasons-and-stripes.ipynb)



Browser-only piplite prepares numpy, pandas, matplotlib and Plotly 6.5.0; native kernels use installed scientific packages. Local calculations use bundled helpers and inputs; Plotly rendering can require its versioned CDN and WebGL. Seeded monthly examples are synthetic. The C3S warming-level table gives model windows, Natural Earth provides real generalized coastlines, and the CDS catalogue is metadata. No authenticated CDS request, archive download or native regridding is run.

**Provenance:** `content/04-seasons-and-stripes.ipynb` at `b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a`; original SHA-256 `2901a1a3a9676d93411d17e327f0fce7739366e66af05bd8ee743e22c3a2a337`. Original course material MIT; C3S auxiliary table Apache-2.0, Natural Earth public domain and CDS catalogue metadata CC BY 4.0. Preserve source credits, licenses and artwork provenance.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Original climate-globe illustration introducing Annual cycles and seasonal stripes; illustrative, not a data map](assets/atlas-splash.png)

# 04 · Annual cycles and seasonal stripes

**Guiding question:** Does an annual average conceal which months are changing?

**Data:** Synthetic teaching examples  
**Time:** 20–35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.  
**Author:** JupyterLite-C3S-Atlas contributors (AI assisted) · **Created/updated:** 2026-11-15  
**Inputs:** bundled reference files and/or seeded examples specified below. **Outputs:** in-memory tables and figures; source data are not modified.  
**Runtime:** Python/Pyodide; NumPy, pandas, Matplotlib, Plotly; versions and tests are recorded in the validation report.  
**Change log:** initial educational chapter. **Contact:** [repository issues](https://github.com/jltobias/JupyterLite-C3S-Atlas/issues).

[Open in JupyterLite Lab](https://jltobias.github.io/JupyterLite-C3S-Atlas/lite/lab/index.html?path=04-seasons-and-stripes.ipynb) · [Official Atlas](https://atlas.climate.copernicus.eu/)


## Read the evidence

A seasonal cycle keeps the calendar month visible. Seasonal stripes are a
       year-by-month anomaly matrix with one colour scale across all cells. Choose a
       diverging palette centred at zero for signed anomalies and retain a numerical legend.

       A season crossing New Year needs an explicit season-year rule. For DJF we assign
       December to the following year and exclude incomplete seasons. Seasonal counts
       should be summed; monthly extremes need max/min aggregation, not a mean.

## Start the browser lab

Run this setup cell once. First load needs internet for the Python runtime and packages. The remaining lesson uses bundled data. If a 3D view is unavailable, use the 2D/table alternative. Download your edited notebook before clearing browser storage.

In [1]:
# Install only in the browser kernel; desktop dependencies come from requirements.txt.
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "plotly==6.5.0"])

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from atlas_helpers import (
    DATA, SYNTHETIC, COLORS, show, monthly_demo, anomalies, regular_grid,
    area_mean, coastlines, map_figure, globe_figure, warming_levels, load_monthly_csv,
)
print("Ready. Browser edits live in this browser; download work you want to keep.")


Ready. Browser edits live in this browser; download work you want to keep.


## Compare monthly normals

In [2]:
df = monthly_demo()
fig = go.Figure()
for start, end in [(1961, 1990), (1991, 2020)]:
    cycle = df[df.year.between(start, end)].groupby("month").t.mean()
    fig.add_scatter(x=cycle.index, y=cycle.values, mode="lines+markers", name=f"{start}–{end}")
fig.update_layout(title="Seasonal cycle · SYNTHETIC", xaxis_title="Month", yaxis_title="Mean temperature (°C)")
show(fig)

Figure({
    'data': [{'mode': 'lines+markers',
              'name': '1961–1990',
              'type': 'scatter',
              'x': {'bdata': 'AQAAAAIAAAADAAAABAAAAAUAAAAGAAAABwAAAAgAAAAJAAAACgAAAAsAAAAMAAAA', 'dtype': 'i4'},
              'y': {'bdata': ('n8lFkkBVEUAB5cS/HP8VQIsMMRPKay' ... 'Ng+HjIKkDGsZuEkN4hQO+xX/YNhBVA'),
                    'dtype': 'f8'}},
             {'mode': 'lines+markers',
              'name': '1991–2020',
              'type': 'scatter',
              'x': {'bdata': 'AQAAAAIAAAADAAAABAAAAAUAAAAGAAAABwAAAAgAAAAJAAAACgAAAAsAAAAMAAAA', 'dtype': 'i4'},
              'y': {'bdata': ('ZoOlQ1uHFEBZ/N0B6HoZQBfs94o4MS' ... '2sHQtELEC6FIBEHTYjQHuhmzoACBlA'),
                    'dtype': 'f8'}}],
    'layout': {'font': {'size': 14},
               'height': 470,
               'margin': {'b': 65, 'l': 65, 'r': 45, 't': 90},
               'template': '...',
               'title': {'text': 'Seasonal cycle · SYNTHETIC'},
               'xaxis': {'title': {'text': 'Month'}},
               'yaxis': {'title': {'text': 'Mean temperature (°C)'}}}
})

## Expose month-by-month changes

In [3]:
df["anomaly"] = anomalies(df, "t", (1961, 1990))
matrix = df.pivot(index="year", columns="month", values="anomaly")
limit = np.abs(matrix.to_numpy()).max()
fig = go.Figure(go.Heatmap(x=matrix.columns, y=matrix.index, z=matrix.to_numpy(),
    colorscale="RdBu_r", zmin=-limit, zmax=limit, colorbar=dict(title="Δ °C")))
fig.update_layout(title="Seasonal stripes · SYNTHETIC", xaxis_title="Month", yaxis_title="Year")
show(fig)
# Require all three months; dropping absent months can bias a seasonal mean.
djf = df[df.month.isin([12, 1, 2])].copy()
djf["season_year"] = djf.year + (djf.month == 12).astype(int)
seasons = djf.groupby("season_year").agg(mean_t=("t", "mean"), n=("t", "count"))
display(seasons[seasons.n == 3].head())

Figure({
    'data': [{'colorbar': {'title': {'text': 'Δ °C'}},
              'colorscale': [[0.0, 'rgb(5,48,97)'], [0.1, 'rgb(33,102,172)'],
                             [0.2, 'rgb(67,147,195)'], [0.3, 'rgb(146,197,222)'],
                             [0.4, 'rgb(209,229,240)'], [0.5, 'rgb(247,247,247)'],
                             [0.6, 'rgb(253,219,199)'], [0.7, 'rgb(244,165,130)'],
                             [0.8, 'rgb(214,96,77)'], [0.9, 'rgb(178,24,43)'],
                             [1.0, 'rgb(103,0,31)']],
              'type': 'heatmap',
              'x': {'bdata': 'AQAAAAIAAAADAAAABAAAAAUAAAAGAAAABwAAAAgAAAAJAAAACgAAAAsAAAAMAAAA', 'dtype': 'i4'},
              'y': {'bdata': ('qQcAAKoHAACrBwAArAcAAK0HAACuBw' ... 'cAAOAHAADhBwAA4gcAAOMHAADkBwAA'),
                    'dtype': 'i4'},
              'z': {'bdata': ('IFoCr+pIzz8YrF+m4M7nv6DUen0Wtd' ... 'Z9pPHc4T/QwyGeU/TkP8DMblYNzvk/'),
                    'dtype': 'f8',
                    'shape': '60, 12'},
              'zmax': np.float64(2.697761444299946),
              'zmin': np.float64(-2.697761444299946)}],
    'layout': {'font': {'size': 14},
               'height': 470,
               'margin': {'b': 65, 'l': 65, 'r': 45, 't': 90},
               'template': '...',
               'title': {'text': 'Seasonal stripes · SYNTHETIC'},
               'xaxis': {'title': {'text': 'Month'}},
               'yaxis': {'title': {'text': 'Year'}}}
})

,mean_t,n
season_year,,
1962,4.562138,3
1963,4.464624,3
1964,5.113176,3
1965,4.896526,3
1966,5.019298,3


## Try it and explain

Which DJF seasons are incomplete at the ends of the record, and why?

**Interpretation check:** DJF 1961 lacks December 1960; DJF 2021 lacks January and February 2021. Only 1962–2020 contain all three months.

## Transfer to the official Atlas

Compare the annual cycle and seasonal stripes for one region. Keep dataset, reference, baseline, and variable fixed.

Record the selections, source version, units, periods, and uncertainty legend. These independent lessons do not duplicate every production workflow in the official tools.

## Sources and credit

- [C3S Atlas dataset and DOI](https://doi.org/10.24381/cds.h35hb680)
- [CSIC / Predictia Atlas user guide](https://confluence.ecmwf.int/spaces/CKB/pages/394237545/Copernicus+Interactive+Climate+Atlas+User+Guide)
- [Official C3S Atlas User Tools and contributors](https://ecmwf-projects.github.io/c3s-atlas/)

Original teaching text and code; original AI-generated splash illustration. Real reference files retain their source licences. Synthetic examples are not Copernicus measurements. See [credits](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/credits.md) and [data guide](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/data-guide.md).

**Continue:** [Heat, cold, and threshold indices](05-heat-and-cold.ipynb)